In [3]:
import pandas as pd
import numpy as np

df_15_min = pd.read_csv(
    "C:/Users/User/Desktop/AlphaMath-QuantCore/Backtest/Csvs/NQ1!_MAIN_15M_NYC_with_holidays.csv"
)
df_1D = pd.read_csv(
    "C:/Users/User/Desktop/AlphaMath-QuantCore/Backtest/Csvs/NQ1!_MAIN_1D.csv"
)
df_1_min = pd.read_csv(
    "C:/Users/User/Desktop/AlphaMath-QuantCore/Backtest/Csvs/NQ1!_MAIN_1M_NYC.csv"
)

df_1_min["datetime"] = pd.to_datetime(df_1_min["datetime"], errors="coerce")
df_15_min["datetime"] = pd.to_datetime(df_15_min["datetime"])
df_1D["datetime"] = pd.to_datetime(df_1D["datetime"])


def calculate_overnight_range_pct(df_input):

    # Make a copy to avoid modifying original
    df_copy = df_input.copy()

    # Create masks for the time range (16:00 to 9:15 next day)
    df_copy["time"] = df_copy["datetime"].apply(lambda x: x.time())
    start_time = pd.Timestamp("16:00:00").time()
    end_time = pd.Timestamp("09:15:00").time()

    # Create session groups to handle overnight sessions
    df_copy["date"] = df_copy["datetime"].apply(lambda x: x.date())
    df_copy["session_date"] = df_copy["date"]
    # Adjust session date for times before 9:15 (they belong to previous session)
    df_copy.loc[df_copy["time"] <= end_time, "session_date"] = df_copy.loc[
        df_copy["time"] <= end_time, "date"
    ] - pd.Timedelta(days=1)

    # Filter for the time range
    mask = (df_copy["time"] >= start_time) | (df_copy["time"] <= end_time)
    session_data = df_copy[mask].copy()

    # Calculate session high and low for each session
    session_stats = session_data.groupby("session_date").agg(
        {"high": "max", "low": "min"}
    )

    # Calculate percentage range: (high - low) / low * 100
    session_range_pct = (
        (session_stats["high"] - session_stats["low"]) / session_stats["low"] * 100
    )

    # Map back to original dataframe
    df_copy["overnight_range_pct"] = df_copy["session_date"].map(session_range_pct)

    # Clean up temporary columns
    df_copy.drop(["time", "date", "session_date"], axis=1, inplace=True)

    return df_copy


def calculate_atr(df_input, period=20):
    """Calculate Average True Range (ATR) for given period"""
    df_copy = df_input.copy()

    # Calculate True Range components as percentages
    df_copy["hl"] = (df_copy["high"] - df_copy["low"]) / df_copy["low"] * 100
    df_copy["hc"] = abs(
        (df_copy["high"] - df_copy["close"].shift(1)) / df_copy["close"].shift(1) * 100
    )
    df_copy["lc"] = abs(
        (df_copy["low"] - df_copy["close"].shift(1)) / df_copy["close"].shift(1) * 100
    )

    # True Range is the maximum of the three components
    df_copy["tr"] = df_copy[["hl", "hc", "lc"]].max(axis=1)

    # Calculate ATR using rolling mean
    df_copy["atr"] = df_copy["tr"].rolling(window=period).mean().shift(1)

    return df_copy[["atr"]]


def add_daily_atr_to_intraday(df_intraday, df_daily):
    """Add daily ATR values to intraday dataframe at 9:30 times"""
    # Calculate ATR for daily data
    daily_atr = calculate_atr(df_daily, period=20)

    # Create a copy of intraday data
    df_result = df_intraday.copy()

    # Extract date and time for intraday data
    df_result["date"] = df_result["datetime"].apply(lambda x: x.date())
    df_result["time"] = df_result["datetime"].apply(lambda x: x.time())

    # Create ATR column initialized with NaN
    df_result["daily_atr"] = np.nan

    # Find 9:30 rows and map ATR values
    target_time = pd.Timestamp("09:30:00").time()
    mask_930 = df_result["time"] == target_time

    # Create mapping from daily ATR
    atr_mapping = daily_atr["atr"].to_dict()
    # build a date→ATR mapping from the daily df's datetime column
    date_to_atr = pd.Series(
        daily_atr["atr"].values, index=df_daily["datetime"].dt.date
    ).to_dict()

    # Apply ATR values to 9:30 rows
    df_result.loc[mask_930, "daily_atr"] = df_result.loc[mask_930, "date"].map(
        date_to_atr
    )

    # Clean up temporary columns
    df_result.drop(["date", "time"], axis=1, inplace=True)

    return df_result


# Apply the function


def add_daily_volume_avg_to_intraday(df_intraday, df_daily):
    """Add daily 20-period volume average to intraday dataframe at 9:30 times"""
    # Calculate 20-period rolling average of volume for daily data
    df_daily_copy = df_daily.copy()
    df_daily_copy["volume_20avg"] = (
        df_daily_copy["volume"].rolling(window=20).mean().shift(1)
    )

    # Create a copy of intraday data
    df_result = df_intraday.copy()

    # Extract date and time for intraday data
    df_result["date"] = df_result["datetime"].apply(lambda x: x.date())
    df_result["time"] = df_result["datetime"].apply(lambda x: x.time())

    # Create volume avg column initialized with NaN
    df_result["daily_volume_20avg"] = np.nan

    # Find 9:30 rows
    target_time = pd.Timestamp("09:30:00").time()
    mask_930 = df_result["time"] == target_time

    # Create mapping from daily volume average
    volume_mapping = df_daily_copy["volume_20avg"].to_dict()
    # Create mapping from daily volume average using the datetime index
    volume_mapping = df_daily_copy.set_index("datetime")["volume_20avg"].to_dict()
    date_to_volume = {dt.date(): vol for dt, vol in volume_mapping.items()}

    # Apply volume average values to 9:30 rows
    df_result.loc[mask_930, "daily_volume_20avg"] = df_result.loc[mask_930, "date"].map(
        date_to_volume
    )

    # Clean up temporary columns
    df_result.drop(["date", "time"], axis=1, inplace=True)

    return df_result


def calculate_overnight_return_pct(df_input):

    # Make a copy to avoid modifying original
    df_copy = df_input.copy()
    # extract time and date
    df_copy["date"] = df_copy["datetime"].apply(lambda x: x.date())
    df_copy["time"] = df_copy["datetime"].apply(lambda x: x.time())

    # define overnight session window: 16:00 → 09:15
    start_time = pd.Timestamp("16:00:00").time()
    end_time = pd.Timestamp("09:15:00").time()

    # assign each bar to its overnight session_date
    df_copy["session_date"] = df_copy["date"]
    mask_early = df_copy["time"] <= end_time
    df_copy.loc[mask_early, "session_date"] = df_copy.loc[
        mask_early, "date"
    ] - pd.Timedelta(days=1)

    # filter only overnight bars
    mask_session = (df_copy["time"] >= start_time) | (df_copy["time"] <= end_time)
    session_data = df_copy[mask_session]

    # compute first & last close per session
    stats = session_data.groupby("session_date")["close"].agg(
        first_close="first", last_close="last"
    )
    stats["overnight_return_pct"] = (
        (stats["last_close"] - stats["first_close"]) / stats["first_close"] * 100
    )
    session_ret = stats["overnight_return_pct"].to_dict()

    # map return back, but only show it at the 09:45 bar
    df_copy["overnight_return_pct"] = df_copy["session_date"].map(session_ret)
    target_time = pd.Timestamp("09:30:00").time()
    df_copy.loc[df_copy["time"] != target_time, "overnight_return_pct"] = np.nan

    # cleanup
    df_copy.drop(["time", "date", "session_date"], axis=1, inplace=True)

    return df_copy


def robust_outlier_sensitive_index(series, alpha=1.0, beta=2.0, threshold=3.0):
    series = series.dropna()  # handle missing values
    if series.empty:
        return np.nan

    median = series.median()
    mad = np.median(np.abs(series - median))

    # Avoid divide-by-zero if MAD=0
    mad = mad if mad > 0 else 1e-9

    # Count outliers beyond threshold × MAD
    deviations = np.abs(series - median)
    outliers = (deviations > threshold * mad).sum()
    p_outliers = outliers / len(series)

    # Stability score
    score = 1 / (1 + alpha * mad + beta * p_outliers)
    return score


def calculate_opening_range_stability(df_input):

    df_copy = df_input.copy()

    # Extract date and time
    df_copy["date"] = df_copy["datetime"].apply(lambda x: x.date())
    df_copy["time"] = df_copy["datetime"].apply(lambda x: x.time())

    # Initialize the new column
    df_copy["opening_range_stability"] = np.nan

    # Define time range for opening period
    start_time = pd.Timestamp("09:30:00").time()
    end_time = pd.Timestamp("09:44:00").time()

    # Filter for opening range times
    mask_opening = (df_copy["time"] >= start_time) & (df_copy["time"] <= end_time)
    opening_data = df_copy[mask_opening].copy()

    if opening_data.empty:
        return df_copy.drop(["date", "time"], axis=1)

    # Calculate true range for opening data
    opening_data["hl"] = opening_data["high"] - opening_data["low"]
    opening_data["hc"] = abs(opening_data["high"] - opening_data["close"].shift(1))
    opening_data["lc"] = abs(opening_data["low"] - opening_data["close"].shift(1))
    opening_data["tr"] = opening_data[["hl", "hc", "lc"]].max(axis=1)

    # Group by date and calculate stability for each day
    for date, group in opening_data.groupby("date"):
        tr_series = group["tr"]
        if len(tr_series) > 0:
            stability_score = robust_outlier_sensitive_index(tr_series)

            # Find matching rows in df_15_min for this date and 9:30 time
            df_15_min_mask = (
                df_15_min["datetime"].apply(lambda x: x.date()) == date
            ) & (df_15_min["datetime"].apply(lambda x: x.time()) == start_time)
            df_15_min.loc[df_15_min_mask, "opening_range_stability"] = stability_score



def calculate_opening_range_breakout(df_input):
    """
    Calculate 15-minute ORB (09:30–09:44) and evaluate:
      - break_direction: first break of the opening range (+1 / -1 / 0)
      - breakout_success: 1 if target hit; else MFE / opening_range until stop/time
      - break_time: timestamp of the first break
      - mfe_to_crossback_ratio: MFE / opening_range until first cross-back past breakout point (or 15:59)
      - mfe_to_halfstop_ratio: MFE / opening_range until first retrace = half the opening range toward stop (or 15:59)

    Notes:
      * Results are written to the FIRST bar inside the opening window (more robust than assuming 09:30 exists).
      * Tie-handling inside a 1-minute bar is optimistic: we count favorable highs/lows before checking reversals.
      * Date/time extraction is done via .apply(lambda x: x.date()) / .time(), as requested.
    """
    df_copy = df_input.copy()

    # Basic checks and ordering
    if "datetime" not in df_copy.columns:
        raise ValueError("df_input must contain a 'datetime' column.")
    for col in ("high", "low"):
        if col not in df_copy.columns:
            raise ValueError(f"df_input must contain '{col}' column.")
    df_copy = df_copy.sort_values("datetime").reset_index(drop=True)

    # Extract date and time (per your requirement)
    df_copy["_date"] = df_copy["datetime"].apply(lambda x: x.date())
    df_copy["_time"] = df_copy["datetime"].apply(lambda x: x.time())

    # Output columns
    df_copy["break_direction"] = np.nan
    df_copy["breakout_success"] = np.nan
    df_copy["break_time"] = pd.NaT
    df_copy["mfe_to_crossback_ratio"] = np.nan
    df_copy["mfe_to_halfstop_ratio"] = np.nan

    # Time bounds
    opening_start = pd.Timestamp("09:30:00").time()
    opening_end   = pd.Timestamp("09:44:00").time()   # inclusive
    test_start    = pd.Timestamp("09:45:00").time()
    test_end      = pd.Timestamp("15:59:00").time()

    # Iterate per trading day
    for day, g in df_copy.groupby("_date", sort=False):
        # Opening window
        opening_mask = (g["_time"] >= opening_start) & (g["_time"] <= opening_end)
        opening_data = g.loc[opening_mask]
        if opening_data.empty:
            continue

        opening_high = opening_data["high"].max()
        opening_low  = opening_data["low"].min()
        opening_range = opening_high - opening_low

        # Test window
        test_mask = (g["_time"] >= test_start) & (g["_time"] <= test_end)
        test_data = g.loc[test_mask]
        if test_data.empty:
            continue

        # First break detection
        break_direction = 0
        break_row = None
        for _, row in test_data.iterrows():
            if row["high"] > opening_high:
                break_direction = 1
                break_row = row
                break
            elif row["low"] < opening_low:
                break_direction = -1
                break_row = row
                break

        # Write to first bar in opening window
        write_idx = opening_data.index[0]
        df_copy.loc[write_idx, "break_direction"] = break_direction

        # No break -> results zero/NaT
        if break_direction == 0:
            df_copy.loc[write_idx, "breakout_success"] = 0.0
            df_copy.loc[write_idx, "break_time"] = pd.NaT
            df_copy.loc[write_idx, "mfe_to_crossback_ratio"] = 0.0
            df_copy.loc[write_idx, "mfe_to_halfstop_ratio"] = 0.0
            continue

        # Levels based on direction
        if break_direction == 1:
            # Up-break
            target_level   = opening_high + opening_range
            stop_level     = opening_low
            break_point    = opening_high
            halfstop_level = break_point - opening_range / 2.0
            # conditions
            def favorable_adv(r): return max(0.0, r["high"] - break_point)
            def target_hit(r):    return r["high"] >= target_level
            def stop_hit(r):      return r["low"] <= stop_level
            def crossback(r):     return r["low"] < break_point       # passed below breakout
            def halfstop(r):      return r["low"] <= halfstop_level   # 50% retrace toward stop
        else:
            # Down-break
            target_level   = opening_low - opening_range
            stop_level     = opening_high
            break_point    = opening_low
            halfstop_level = break_point + opening_range / 2.0
            # conditions
            def favorable_adv(r): return max(0.0, break_point - r["low"])
            def target_hit(r):    return r["low"] <= target_level
            def stop_hit(r):      return r["high"] >= stop_level
            def crossback(r):     return r["high"] > break_point      # passed above breakout
            def halfstop(r):      return r["high"] >= halfstop_level  # 50% retrace toward stop

        # From (and including) the break bar to end of test window
        break_tod = break_row["_time"]
        onward_mask = (g["_time"] >= break_tod) & (g["_time"] <= test_end)
        onward = g.loc[onward_mask]

        # Record the exact timestamp of the break
        df_copy.loc[write_idx, "break_time"] = df_copy.loc[int(break_row.name), "datetime"]

        # Denominator for ratios (target distance)
        denom = opening_range if opening_range > 0 else None

        # Track three paths simultaneously:
        # 1) "General" success: stop/target logic (your original behavior)
        # 2) MFE until cross-back
        # 3) MFE until half-stop retrace
        mfe_general = 0.0
        breakout_success = 0.0

        mfe_cb = 0.0            # until crossback
        cb_done = False

        mfe_half = 0.0          # until half-stop
        half_done = False

        for _, r in onward.iterrows():
            # Update favorable moves
            adv = favorable_adv(r)
            if adv > mfe_general:
                mfe_general = adv
            if not cb_done and adv > mfe_cb:
                mfe_cb = adv
            if not half_done and adv > mfe_half:
                mfe_half = adv

            # --- General path (target/stop with optimistic tie: target before stop) ---
            if target_hit(r):
                breakout_success = 1.0
                # We do NOT break here because we still need to finish cb/half tracking to EOD or threshold.
                # However, if you prefer to stop the whole scan after target, you could break.
            elif stop_hit(r) and breakout_success < 1.0:
                breakout_success = (min(mfe_general / denom, 1.0) if denom else 0.0)
                # continue scanning for cb/half metrics

            # --- Cross-back metric: stop tracking once crossback happens ---
            if not cb_done and crossback(r):
                cb_done = True  # finalize mfe_cb at its current value

            # --- Half-stop metric: stop tracking once half retrace happens ---
            if not half_done and halfstop(r):
                half_done = True  # finalize mfe_half at its current value

            # If both cb and half are done AND general already decided as 1.0, 
            # we could break for speed; but we keep full scan for clarity.

        # If neither target nor stop hit by 15:59, general success is MFE/denom
        if breakout_success < 1.0:
            breakout_success = (min(mfe_general / denom, 1.0) if denom else 0.0)

        # Normalize new metrics (if denom==0 -> 0.0)
        mfe_cb_ratio = (min(mfe_cb / denom, 1.0) if denom else 0.0)
        mfe_half_ratio = (min(mfe_half / denom, 1.0) if denom else 0.0)

        # Write results
        df_copy.loc[write_idx, "breakout_success"] = breakout_success
        df_copy.loc[write_idx, "mfe_to_crossback_ratio"] = mfe_cb_ratio
        df_copy.loc[write_idx, "mfe_to_halfstop_ratio"] = mfe_half_ratio

    # Cleanup
    df_copy.drop(columns=["_date", "_time"], inplace=True)
    return df_copy



# Apply the opening range stability function
calculate_opening_range_stability(df_1_min)

# Apply the opening range breakout function
df_1_min = calculate_opening_range_breakout(df_1_min)

# Transfer the breakout data to df_15_min at 9:30 times
df_15_min["break_direction"] = np.nan
df_15_min["breakout_success"] = np.nan
df_15_min["mfe_to_crossback_ratio"] = np.nan
df_15_min["mfe_to_halfstop_ratio"] = np.nan

# Extract time for df_15_min
df_15_min["time_temp"] = df_15_min["datetime"].apply(lambda x: x.time())
df_15_min["date_temp"] = df_15_min["datetime"].apply(lambda x: x.date())

# Extract corresponding data from df_1_min
df_1_min["time_temp"] = df_1_min["datetime"].apply(lambda x: x.time())
df_1_min["date_temp"] = df_1_min["datetime"].apply(lambda x: x.date())

# Get 9:30 data from df_1_min
target_time = pd.Timestamp("09:30:00").time()
df_1_min_930 = df_1_min[df_1_min["time_temp"] == target_time].copy()

# Create mapping dictionaries
break_direction_map = df_1_min_930.set_index("date_temp")["break_direction"].to_dict()
breakout_success_map = df_1_min_930.set_index("date_temp")["breakout_success"].to_dict()
mfe_to_crossback_ratio_map = df_1_min_930.set_index("date_temp")["mfe_to_crossback_ratio"].to_dict()
mfe_to_halfstop_ratio_map = df_1_min_930.set_index("date_temp")["mfe_to_halfstop_ratio"].to_dict()

# Apply to df_15_min at 9:30 times
mask_930_15min = df_15_min["time_temp"] == target_time
df_15_min.loc[mask_930_15min, "break_direction"] = df_15_min.loc[
    mask_930_15min, "date_temp"
].map(break_direction_map)
df_15_min.loc[mask_930_15min, "breakout_success"] = df_15_min.loc[
    mask_930_15min, "date_temp"
].map(breakout_success_map)
df_15_min.loc[mask_930_15min, "mfe_to_crossback_ratio"] = df_15_min.loc[
    mask_930_15min, "date_temp"
].map(mfe_to_crossback_ratio_map)
df_15_min.loc[mask_930_15min, "mfe_to_halfstop_ratio"] = df_15_min.loc[
    mask_930_15min, "date_temp"
].map(mfe_to_halfstop_ratio_map)

# Clean up temporary columns
df_15_min.drop(["time_temp", "date_temp"], axis=1, inplace=True)
df_1_min.drop(["time_temp", "date_temp"], axis=1, inplace=True)


calculate_opening_range_stability(df_1_min)

df_15_min["range"] = df_15_min["high"] / df_15_min["low"]


df_15_min = add_daily_volume_avg_to_intraday(df_15_min, df_1D)

df_15_min = add_daily_atr_to_intraday(df_15_min, df_1D)

df_15_min = calculate_overnight_range_pct(df_15_min)

df_15_min = calculate_overnight_return_pct(df_15_min)

df_15_min["atr"] = calculate_atr(df_15_min, period=20)
df_15_min["atr"] = df_15_min["atr"].shift(1)
df_15_min.rename(columns={"atr": "intraday_atr"}, inplace=True)

# Add calculations for 9:30 rows
df_15_min["time"] = df_15_min["datetime"].apply(lambda x: x.time())
target_time = pd.Timestamp("09:30:00").time()
mask_930 = df_15_min["time"] == target_time

# Initialize new columns
df_15_min["overnight_range_over_daily_atr"] = np.nan
df_15_min["range_over_intraday_atr"] = np.nan
df_15_min["volume_over_daily_avg"] = np.nan

# Calculate ratios for 9:30 rows
df_15_min.loc[mask_930, "overnight_range_over_daily_atr"] = (
    df_15_min.loc[mask_930, "overnight_range_pct"]
    / df_15_min.loc[mask_930, "daily_atr"]
)
df_15_min.loc[mask_930, "range_over_intraday_atr"] = (
    df_15_min.loc[mask_930, "range"] / df_15_min.loc[mask_930, "intraday_atr"]
)
df_15_min.loc[mask_930, "volume_over_daily_avg"] = (
    df_15_min.loc[mask_930, "volume"] / df_15_min.loc[mask_930, "daily_volume_20avg"]
)

# Clean up temporary column
df_15_min.drop("time", axis=1, inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_41520\3889940044.py:14: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_1_min["datetime"] = pd.to_datetime(df_1_min["datetime"], errors="coerce")
C:\Users\User\AppData\Local\Temp\ipykernel_41520\3889940044.py:15: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_15_min["datetime"] = pd.to_datetime(df_15_min["datetime"])
C:\Users\User\AppData\Local\Temp\ipykernel_41520\3889940044.py:383: FutureWarning: 

In [4]:

df_15_min[mask_930].to_csv('C:/Users/User/Desktop/AlphaMath-QuantCore/Backtest/Csvs/NQ1!_15M_final_processed.csv')